In [2]:
import re
import os
import pandas as pd
import numpy as np

### tabular data generation

In [ ]:
df = pd.read_csv(f'./data/test_kcat.csv')
df.iloc[0]


EntryID                                                              BRENDA_46
UniprotID                                                           A0A023DFE8
Substrate                               N-3-oxododecanoyl-L-homoserine lactone
kcat                                                                      9.48
Temperature                                                               25.0
pH                                                                         8.3
Organism                                     Saccharococcus caldoxylosilyticus
ECNumber                                                              3.1.1.81
Mutation                                                              wildtype
SMILES                                      CCCCCCCCCC(=O)CC(=O)N[C@H]1CCOC1=O
Sequence                     MANVIKARPKLYVMDNGRMRMDKNWMIAMHNPATIHNPNAQTEFVE...
StructureFile                                    AF-A0A023DFE8-F1-model_v4.pdb
LmdbKey                                             

### tabular data process

In [ ]:
df['T_K'] = df['Temperature'].apply(lambda x: float(x) + 273.15 if x != '-' else None)
df['inv_T'] = df['T_K'].apply(lambda x: 1.0 / x if x is not None else None)
df['log_T'] = df['T_K'].apply(lambda x: np.log(x) if x is not None else None)

def clean_ph(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        x = x.rstrip('.')      
        x = x.replace('..', '.')  
        try:
            x = float(x)
        except:
            pass             
    return x

df['pH'] = df['pH'].apply(clean_ph)
df['pH_centered'] = df['pH'].apply(lambda x: x - 7.0 if x != '-' else None)
df['pH_squared'] = df['pH_centered'].apply(lambda x: x**2 if x != '-' else None)



In [ ]:
def split_ec(ec):
    parts = str(ec).split('.')
    while len(parts) < 4:
        parts.append('0')
    return parts[:4]

ecs = df['ECNumber'].apply(split_ec)
df[['ec1','ec2','ec3','ec4']] = pd.DataFrame(ecs.tolist(), index=df.index)


In [ ]:
AA_PROPS = {
    'A': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=1.8),
    'R': dict(charge=1, is_polar=True,  is_aromatic=False, hydropathy=-4.5),
    'N': dict(charge=0, is_polar=True,  is_aromatic=False, hydropathy=-3.5),
    'D': dict(charge=-1,is_polar=True,  is_aromatic=False, hydropathy=-3.5),
    'C': dict(charge=0, is_polar=True,  is_aromatic=False, hydropathy=2.5),
    'Q': dict(charge=0, is_polar=True,  is_aromatic=False, hydropathy=-3.5),
    'E': dict(charge=-1,is_polar=True,  is_aromatic=False, hydropathy=-3.5),
    'G': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=-0.4),
    'H': dict(charge=1, is_polar=True,  is_aromatic=False, hydropathy=-3.2),
    'I': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=4.5),
    'L': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=3.8),
    'K': dict(charge=1, is_polar=True,  is_aromatic=False, hydropathy=-3.9),
    'M': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=1.9),
    'F': dict(charge=0, is_polar=False, is_aromatic=True,  hydropathy=2.8),
    'P': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=-1.6),
    'S': dict(charge=0, is_polar=True,  is_aromatic=False, hydropathy=-0.8),
    'T': dict(charge=0, is_polar=True,  is_aromatic=False, hydropathy=-0.7),
    'W': dict(charge=0, is_polar=False, is_aromatic=True,  hydropathy=-0.9),
    'Y': dict(charge=0, is_polar=True,  is_aromatic=True,  hydropathy=-1.3),
    'V': dict(charge=0, is_polar=False, is_aromatic=False, hydropathy=4.2),
}

def parse_mutation_string(mutation_str):
    if not isinstance(mutation_str, str):
        return []
    
    s = mutation_str.strip()
    if s == "" or s.lower() in ["wt", "wildtype", "wild-type"]:
        return []
    
    mutations = []
    parts = s.split('/')
    pattern = re.compile(r'^([A-Z])(\d+)([A-Z])$')
    for part in parts:
        part = part.strip()
        if not part:
            continue
        m = pattern.match(part)
        if m:
            from_aa = m.group(1)
            pos = int(m.group(2))
            to_aa = m.group(3)
            mutations.append((from_aa, pos, to_aa))
        else:
            # 其他格式暂时忽略，也可以在此打印日志
            pass
    return mutations

def compute_mutation_features_for_seq(seq, mutation_str):
    L = len(seq) if isinstance(seq, str) else 0
    muts = parse_mutation_string(mutation_str)
    
    features = {
        "num_mutations": 0,
        "mut_ratio": 0.0,
        "min_rel_pos": 0.0,
        "max_rel_pos": 0.0,
        "mean_rel_pos": 0.0,
        "std_rel_pos": 0.0,
        "frac_mut_Nterm": 0.0,
        "frac_mut_Cterm": 0.0,
        "num_charge_change": 0,
        "any_charge_change": 0,
        "net_charge_change": 0.0,
        "abs_net_charge_change": 0.0,
        "num_polarity_change": 0,
        "net_hydropathy_change": 0.0,
        "abs_net_hydropathy_change": 0.0,
        "num_hydro_increase": 0,
        "num_hydro_decrease": 0,
        "num_aromatic_change": 0,
        "any_aromatic_change": 0,
        "num_to_proline": 0,
        "num_from_proline": 0,
        "any_to_proline": 0,
        "any_from_proline": 0,
        "num_to_gly": 0,
        "num_from_gly": 0,
        "any_to_gly": 0,
        "any_from_gly": 0,
    }
    
    if L == 0 or len(muts) == 0:
        return features
    
    rel_positions = []
    nterm_count = 0
    cterm_count = 0
    
    net_charge_change = 0.0
    net_hydro_change = 0.0
    
    charge_change_count = 0
    polarity_change_count = 0
    hydro_increase_count = 0
    hydro_decrease_count = 0
    aromatic_change_count = 0
    
    to_pro_count = 0
    from_pro_count = 0
    to_gly_count = 0
    from_gly_count = 0

    for (from_aa, pos, to_aa) in muts:
        if L > 1:
            rel_pos = (pos - 1) / (L - 1)
        else:
            rel_pos = 0.0
        rel_positions.append(rel_pos)
        if rel_pos < 0.1:
            nterm_count += 1
        if rel_pos > 0.9:
            cterm_count += 1
        if from_aa not in AA_PROPS or to_aa not in AA_PROPS:
            continue
        
        fa = AA_PROPS[from_aa]
        ta = AA_PROPS[to_aa]
        
        delta_charge = ta["charge"] - fa["charge"]
        net_charge_change += delta_charge
        if delta_charge != 0:
            charge_change_count += 1
        
        if ta["is_polar"] != fa["is_polar"]:
            polarity_change_count += 1
        
        delta_hydro = ta["hydropathy"] - fa["hydropathy"]
        net_hydro_change += delta_hydro
        if delta_hydro > 0:
            hydro_increase_count += 1
        elif delta_hydro < 0:
            hydro_decrease_count += 1
        
        if ta["is_aromatic"] != fa["is_aromatic"]:
            aromatic_change_count += 1
        
        if to_aa == "P":
            to_pro_count += 1
        if from_aa == "P":
            from_pro_count += 1
        if to_aa == "G":
            to_gly_count += 1
        if from_aa == "G":
            from_gly_count += 1
    
    num_mut = len(muts)
    features["num_mutations"] = num_mut
    features["mut_ratio"] = num_mut / L
    
    rel_positions = np.array(rel_positions, dtype=float)
    features["min_rel_pos"] = float(rel_positions.min())
    features["max_rel_pos"] = float(rel_positions.max())
    features["mean_rel_pos"] = float(rel_positions.mean())
    features["std_rel_pos"] = float(rel_positions.std()) if num_mut > 1 else 0.0
    
    features["frac_mut_Nterm"] = nterm_count / num_mut
    features["frac_mut_Cterm"] = cterm_count / num_mut
    
    features["num_charge_change"] = charge_change_count
    features["any_charge_change"] = int(charge_change_count > 0)
    features["net_charge_change"] = float(net_charge_change)
    features["abs_net_charge_change"] = float(abs(net_charge_change))
    
    features["num_polarity_change"] = polarity_change_count
    
    features["net_hydropathy_change"] = float(net_hydro_change)
    features["abs_net_hydropathy_change"] = float(abs(net_hydro_change))
    features["num_hydro_increase"] = hydro_increase_count
    features["num_hydro_decrease"] = hydro_decrease_count
    
    features["num_aromatic_change"] = aromatic_change_count
    features["any_aromatic_change"] = int(aromatic_change_count > 0)
    
    features["num_to_proline"] = to_pro_count
    features["num_from_proline"] = from_pro_count
    features["any_to_proline"] = int(to_pro_count > 0)
    features["any_from_proline"] = int(from_pro_count > 0)
    
    features["num_to_gly"] = to_gly_count
    features["num_from_gly"] = from_gly_count
    features["any_to_gly"] = int(to_gly_count > 0)
    features["any_from_gly"] = int(from_gly_count > 0)
    
    return features

def add_mutation_features(df, seq_col="Sequence", mut_col="Mutation"):
    feature_dicts = []
    for _, row in df.iterrows():
        seq = row[seq_col]
        mut = row.get(mut_col, None)
        feats = compute_mutation_features_for_seq(seq, mut)
        feature_dicts.append(feats)
    
    feat_df = pd.DataFrame(feature_dicts)
    df_out = pd.concat([df.reset_index(drop=True), feat_df], axis=1)
    return df_out

df = add_mutation_features(df)

In [ ]:
import pickle
with open('./data/genus2id.pkl', 'rb') as f:
    genus2id = pickle.load(f)

with open('./data/species2id.pkl', 'rb') as f:
    species2id = pickle.load(f)

df['species_id'] = df['Organism'].map(species2id).astype('int64')
df['genus_id'] = df['genus'].map(genus2id).astype('int64')

In [8]:
categories_names = ['ec1', 'ec2', 'ec3', 'ec4', 'species_id','genus_id']
nums_names = ['T_K', 'inv_T', 'log_T', 
       'pH_centered', 'pH_squared', 'pH',
       'num_mutations', 'mut_ratio',
       'min_rel_pos', 'max_rel_pos', 'mean_rel_pos', 'std_rel_pos','frac_mut_Nterm', 'frac_mut_Cterm',
       'num_charge_change','any_charge_change', 'net_charge_change', 'abs_net_charge_change',
       'num_polarity_change', 
       'num_hydro_increase', 'num_hydro_decrease','net_hydropathy_change','abs_net_hydropathy_change'
       'num_aromatic_change', 'any_aromatic_change', 
       'any_to_proline','any_from_proline', 'any_to_gly', 'any_from_gly', 
       ]

### torchdrug data process\

In [ ]:
import pickle
from tqdm import tqdm
from torchdrug import data

struc_file = list(set(df['StructureFile'].tolist()))
fail_strucs = []
for i in tqdm(struc_file):
    if os.path.exists(f"./data/AFDB/processed_proteins/{i.split('.')[0]}.pkl"):
        continue
    struc_path = os.path.join('/data/AFDB/All_Structure', i)
    try:
        protein = data.Protein.from_pdb(struc_path)
        with open(f"./data/AFDB/processed_proteins/{i.split('.')[0]}.pkl", 'wb') as f:
            pickle.dump(protein, f)
    except:
        fail_strucs.append(i)